# Visualização exploratória

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/03_visualizacao_exploratoria.ipynb)
Gráficos são argumentos. Toda figura terá título, descrição e tabela equivalente. Usaremos SVG acessível e offline.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_04'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd,re
from collections import Counter
from IPython.display import SVG,display
dados=pd.read_csv("dados/documentos.csv")
def barras(rotulos,valores,titulo):
 m=max(valores) or 1; corpo=f'<text x="10" y="22">{titulo}</text>'
 for i,(r,v) in enumerate(zip(rotulos,valores)):
  y=40+i*32; z=400*v/m; corpo+=f'<text x="10" y="{y+16}">{r}</text><rect x="120" y="{y}" width="{z}" height="20" fill="#356a7a"/><text x="{130+z}" y="{y+16}">{v:.1f}</text>'
 return SVG(f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{titulo}" width="650" height="{70+32*len(valores)}">{corpo}</svg>')
def pontos(xs,ys,titulo,linha=False):
 xmin,xmax=min(xs),max(xs); ymin,ymax=min(ys),max(ys); ps=[(50+550*(x-xmin)/(xmax-xmin or 1),320-260*(y-ymin)/(ymax-ymin or 1)) for x,y in zip(xs,ys)]; b=f'<text x="10" y="20">{titulo}</text>'
 if linha: b+=f'<polyline points="{" ".join(f"{x},{y}" for x,y in ps)}" fill="none" stroke="#356a7a"/>'
 else:
  for x,y in ps: b+=f'<circle cx="{x}" cy="{y}" r="5" fill="#9a4f37"/>'
 return SVG(f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{titulo}" width="650" height="350">{b}</svg>')

## Barras — categorias

In [ ]:
f=dados.tema.value_counts(); display(barras(f.index,f.values,"Documentos por tema")); f

## Histograma e boxplot — distribuições

Se os limites dos intervalos são $b_0,b_1,\ldots,b_J$, a altura da barra $j$
é a quantidade de valores dentro daquele intervalo:

$$
h_j=\sum_{i=1}^{n}\mathbf{1}(b_{j-1}<x_i\leq b_j).
$$

Alterar os limites $b_j$ pode mudar a forma visível da distribuição, mesmo sem
alterar os documentos. O boxplot retoma $Q_1$, mediana, $Q_3$ e os limites de
1,5 IQR apresentados no Notebook 01. A tabela abaixo oferece os intervalos e o
resumo numérico equivalentes.

In [ ]:
faixas=[0,400,600,800,1000,2200]; h=dados.groupby(pd.cut(dados.palavras,faixas),observed=False).size(); display(barras(h.index.astype(str),h.values,"Histograma da extensão")); q=dados.palavras.quantile([0,.25,.5,.75,1]); sc=lambda x:60+520*(x-q.iloc[0])/(q.iloc[-1]-q.iloc[0]); svg=f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Boxplot da extensão: mínimo {q.iloc[0]}, Q1 {q.iloc[1]}, mediana {q.iloc[2]}, Q3 {q.iloc[3]}, máximo {q.iloc[4]}" width="650" height="180"><line x1="{sc(q.iloc[0])}" y1="90" x2="{sc(q.iloc[4])}" y2="90" stroke="black"/><rect x="{sc(q.iloc[1])}" y="55" width="{sc(q.iloc[3])-sc(q.iloc[1])}" height="70" fill="#b9d4dc" stroke="black"/><line x1="{sc(q.iloc[2])}" y1="55" x2="{sc(q.iloc[2])}" y2="125" stroke="#9a4f37"/></svg>'; display(SVG(svg)); h,dados.palavras.describe()

## Dispersão — relação entre quantitativas
Padrão visual não implica causalidade.

In [ ]:
display(pontos(dados.paginas.tolist(),dados.palavras.tolist(),"Páginas e palavras")); dados[["paginas","palavras"]].head()

## Série temporal

Se $n_t$ documentos pertencem ao ano $t$, a média anual representada pela
linha é:

$$
\bar{x}_t=\frac{1}{n_t}\sum_{i:\,\mathrm{ano}_i=t}x_i.
$$

A fórmula torna visível o denominador anual. A linha conecta agregados dos
documentos disponíveis; ela também reflete a composição do corpus e não demonstra,
por si só, uma mudança histórica contínua.

In [ ]:
media_por_ano = dados.groupby("ano")["palavras"].mean()
display(pontos(media_por_ano.index.tolist(),media_por_ano.values.tolist(),"Média por ano",linha=True))
media_por_ano

## Frequências textuais
Barras preservam valores melhor que nuvem de palavras.

In [ ]:
co=Counter(re.findall(r"[a-záàâãéêíóôõúç]+"," ".join(dados.texto).lower())); stop={"a","o","e","de","do","da","como","em","nas","um","uma","também"}; top=[z for z in co.most_common() if z[0] not in stop][:10]; display(barras([p for p,n in top],[n for p,n in top],"Termos frequentes")); pd.DataFrame(top,columns=["termo","frequencia"])

## Atividade
Produza barras, histograma/boxplot, dispersão ou tempo e frequência textual. Para cada: tabela, descrição, escala, padrão, caso, limite e hipótese. Escreva aqui.